In [ ]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence , RunnableLambda , RunnableParallel , RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = os.getenv("GROQ_API_KEY")  
)




In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

# https://www.youtube.com/watch?v=2MqmA4i0f1A
video_id = "qjPH9njnaVU"
ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)

print(transcript)


transcript_text = ""
transcript_text = "".join(snippet.text for snippet in transcript)
    

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript_text])

for chunk in chunks:
    chunk.metadata = {"video_id" : video_id}

In [ ]:
chunks

In [ ]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings


embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
hf = HuggingFaceEndpointEmbeddings(
    model=embedding_model,
    task="feature-extraction",
    huggingfacehub_api_token = os.getenv("HUGGINGFACEHUB_API_KEY"),
)

In [ ]:
# from langchain_community.vectorstores import FAISS
# vector_store = FAISS.from_documents(chunks, hf) 


# chunks are converted to embeddings and stored in vector store

from langchain_classic.vectorstores import Chroma 
vectorstore = Chroma(persist_directory="./data", embedding_function=hf)


vectorstore.add_documents(chunks)

In [ ]:
retriever = vectorstore.as_retriever(search_type = 'mmr' , search_kwargs = {'k': 5, 'fetch_k': 20, 'lambda_mult': 0.7})

# 1->pure similarity search
# 0.7 -> 70 percent relevance + 30 percent diversity

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer only from the provided transcript context.
If the context is insufficient , just say you don't know.

{context}
Question : {question} \n

Conversation so far : {history}
""",
input_variables=['context', 'question' , 'history']
)

In [ ]:
parser = StrOutputParser()
chain = prompt | model | parser


In [ ]:
def format_docs(context):
    context_text = "\n\n".join(doc.page_content for doc in context)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({

    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough(),
}
)

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
store = {}

def get_session_history(session_id : str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id] 

In [ ]:
chain_with_memory =  RunnableWithMessageHistory(
    runnable = chain,
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

In [ ]:
# Your current setup
# Chain:

# chain = prompt | model | parser
# This handles question + context → LLM → parsed answer.

# This is the part that produces the AI output.

# Parallel Chain:

# parallel_chain = RunnableParallel({
#     "context": retriever | RunnableLambda(format_docs),
#     "question": RunnablePassthrough()
# })
# This handles preparing the inputs:

# "context" → retrieved docs

# "question" → user input

# Produces a dict like: {"context": <retrieved text>, "question": <user question>}

# Final Chain:

# final_chain = parallel_chain | chain
# Combines both: outputs of parallel_chain are fed into chain.

# 2️⃣ Where RunnableWithMessageHistory should go
# Remember the purpose of RunnableWithMessageHistory:

# Track conversation turns: user messages and AI responses.

# Inject past chat history into the prompt.

# Key point:
# The user message (to track in memory) is the input to the LLM (i.e., question that ends up in chain).

# The AI response is the output of chain, not the retriever or parallel chain.

# So, memory only needs to wrap the part of the pipeline that consumes the user input and produces an AI response.

# ✅ Therefore:
# Wrap chain with RunnableWithMessageHistory, not the whole final_chain.

# Why not the whole final_chain?

# The parallel_chain is just preprocessing (retriever → context).

# Wrapping it would track the intermediate “context” output as a message, which you don’t want in chat history.

# 3️⃣ How to wire it
# # Wrap only the chain
# chain_with_memory = RunnableWithMessageHistory(
#     runnable=chain,
#     get_session_history=get_session_history,
#     input_messages_key="question",
#     history_messages_key="history"  # corresponds to {history} in prompt
# )

# # Final conversational pipeline
# final_chain = parallel_chain | chain_with_memory
# Now, the retriever still runs in parallel to fetch context.

# The LLM chain remembers Q/A history automatically.

# 4️⃣ Invocation
# session_id = "user123"
# result = final_chain.invoke(
#     "Tell me about Romanian elections",
#     config={"configurable": {"session_id": session_id}}
# )
# The user passes only the question.

# RunnableWithMessageHistory adds it to memory and injects past messages into the prompt.

# Only the chain part (prompt → LLM → parser) interacts with memory 


# What gets wrapped
# chain = prompt | model | parser is what you wrap with RunnableWithMessageHistory.

# Why? Because this is the part that:

# Takes the user’s question as input.

# Produces the LLM output, which is what you want to store in chat history.

# 2️⃣ What goes into memory
# Only the user’s question (input_messages_key="question")

# Only the LLM output (AI response) gets stored as AIMessage.

# Intermediate outputs like "context" from the retriever are not stored — they are just inputs to help answer the question.

# 3️⃣ How it fits in the full pipeline
# # Step 1: Prepare inputs
# parallel_chain = RunnableParallel({
#     "context": retriever | RunnableLambda(format_docs),
#     "question": RunnablePassthrough()
# })

# # Step 2: Wrap only the LLM chain with memory
# chain_with_memory = RunnableWithMessageHistory(
#     runnable=chain,                     # prompt | model | parser
#     get_session_history=get_session_history,
#     input_messages_key="question",      # user message to track
#     history_messages_key="history"      # placeholder in prompt
# )

# # Step 3: Combine parallel preprocessing + chain
# final_chain = parallel_chain | chain_with_memory
# User passes only the question to final_chain.invoke().

# The retriever fetches context → chain_with_memory sees the question + history → produces answer → answer stored in memory.




In [ ]:
final_chain = parallel_chain | chain_with_memory

In [ ]:
result = final_chain.invoke(
    "Tell me about Romanian elections",
    config={'configurable': {"session_id": "user1"}}
)

# didnt invoke in this way "question" : "tell me about romanian elections" , because our parallel chain has 2 chains one is context and one is question , both of them expect the same input , if we do "question" , then the input only goes to the "question" placeholder and not the context placeholder , thats why need to pass it like this "....."
# retriever inside the "context" in parallel chain want a normal string , 
# SEE we have 2 inputs one is context and one is question 
# if we pass it like this "question" : "...." , input only goes to question placeholder and not context.
# therefore we just need to pass it as a string "..."

In [ ]:
print(result)

In [ ]:
# What to wrap with RunnableWithMessageHistory
# Wrap the smallest chain that:

# Takes the user’s question (or any human message) as input.

# Produces the LLM output (AI response) as output.

# Usually, this is just your prompt | model | parser chain.

# 2️⃣ What not to wrap
# Don’t wrap preprocessing steps like:

# Retriever fetching documents

# Context formatting (RunnableLambda(format_docs))

# Any parallel chains that don’t produce AI messages

# Why?

# These are not conversational turns, so they don’t belong in history.

# Wrapping them can cause messy history or type errors (like the dict.replace() issue you saw).


# THATS WHY WE DIDNT WRAP THE PARALLEL CHAIN AND ONLY WRAPPED THE NORMAL CHAIN 

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

In [ ]:
app = FastAPI()


class Query(BaseModel) :
    UserMessage : str
    session_id : str
    

@app.post("/predict/")
async def LLMQuery(query :  Query):
    response = final_chain.invoke(
        f"{Query.UserMessage}",
        config = {'configurable' : {"session_id" : f"{query.session_id}"}}
    )

In [ ]:
import time
from datasets import Dataset
from ragas.testset import TestsetGenerator
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas import evaluate
from langchain_groq import ChatGroq

ragas_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key= os.getenv("GROQ_API_KEY"),
    temperature=0.1,
    max_tokens=2048
)


test_generator = TestsetGenerator.from_langchain(
    llm=ragas_model,
    embedding_model=hf
)

print("Generating synthetic test set...")
synthetic_testset = test_generator.generate_with_langchain_docs(
    chunks[:10],
    testset_size=3
)

print(f"\nGenerated {len(synthetic_testset.samples)} test samples")


df = synthetic_testset.to_pandas()
print("\nDataFrame columns:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

questions = df['user_input'].tolist() if 'user_input' in df.columns else df['question'].tolist()
ground_truths = df['reference'].tolist() if 'reference' in df.columns else (
    df['ground_truth'].tolist() if 'ground_truth' in df.columns else [""] * len(questions)
)

print(f"\nExtracted {len(questions)} questions")

print("\nGenerating answers...")
answers = []
retrieved_contexts = []

for i, q in enumerate(questions):
    print(f"\nQuestion {i+1}/{len(questions)}:")
    print(f"  {q[:150]}...")
    
    try:

        retrieval_result = parallel_chain.invoke(q)
        context_text = retrieval_result["context"]
        
        answer = chain_with_memory.invoke(
            {"question": q, "context": context_text, "history": ""},
            config={'configurable': {"session_id": f"ragas_{i}"}}
        )
        
        answers.append(answer)
        retrieved_contexts.append([context_text])
        
        print(f"  ✓ Answer: {answer[:150]}...")
        
        time.sleep(2)  
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        answers.append("Unable to generate answer")
        retrieved_contexts.append([""])


print("\n" + "="*60)
print("Preparing RAGAS evaluation dataset...")
print("="*60)

data_for_ragas = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": retrieved_contexts,
    "ground_truth": ground_truths,
})

print(f"\nDataset info:")
print(f"  - Questions: {len(questions)}")
print(f"  - Answers: {len(answers)}")
print(f"  - Contexts: {len(retrieved_contexts)}")
print(f"  - Ground truths: {len(ground_truths)}")


print("\n--- Sample Entry ---")
print(f"Q: {questions[0]}")
print(f"A: {answers[0][:200]}...")
print(f"GT: {ground_truths[0][:200] if ground_truths[0] else 'N/A'}...")

# STEP 5: Evaluate
print("\n" + "="*60)
print("Running RAGAS evaluation...")
print("="*60)

try:
    results = evaluate(
        dataset=data_for_ragas,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_model,
        embeddings=hf,
    )
    
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)
    print(results)
    
 
    results_df = results.to_pandas()
    print("\n--- Detailed Scores ---")
    print(results_df)
    
    
    print("\n--- Average Scores ---")
    for col in results_df.columns:
        if col in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
            avg_score = results_df[col].mean()
            print(f"{col:20s}: {avg_score:.4f}")
    
    # Save results
    results_df.to_csv('ragas_evaluation_results.csv', index=False)
    print("\n✓ Results saved to 'ragas_evaluation_results.csv'")
    
except Exception as e:
    print(f"\n❌ Evaluation failed: {e}")
    import traceback
    traceback.print_exc()
    
    print("\n--- Attempting evaluation with reduced metrics ---")
    try:
        results = evaluate(
            dataset=data_for_ragas,
            metrics=[faithfulness, answer_relevancy],
            llm=ragas_model,
            embeddings=hf,
        )
        print("\n✓ Partial evaluation successful")
        print(results)
        print(results.to_pandas())
    except Exception as e2:
        print(f"❌ Fallback also failed: {e2}")